# ARC Prize 2026 — ARC-AGI-3 Online Research Run

Online Claude-driven research notebook. This is **not** the prize-safe Kaggle submission because official evaluation does not allow internet/API-based systems. Use `submission_offline.ipynb` for the offline controller.

All logic lives in the `agent/` package; this notebook only wires up dependencies, secrets, and the run.

Before running:
1. Add this repo as a Kaggle Dataset named `arc-prize-2026-agent` (or uncomment the `git clone` line below).
2. Create two Kaggle Secrets: `ARC_API_KEY` and `ANTHROPIC_API_KEY`.
3. Enable Internet in the notebook settings for research runs only (required for the Anthropic and ARC APIs).

In [ ]:
# 1. Install runtime dependencies
!pip install -q arc-agi anthropic

In [ ]:
# 2. Make the agent package importable.
# Option A (default): attach this repo as a Kaggle Dataset and point sys.path at it.
# Option B: clone from GitHub (uncomment after pushing).

import os, sys

DATASET_PATH = "/kaggle/input/arc-prize-2026-agent"
if os.path.isdir(DATASET_PATH):
    sys.path.insert(0, DATASET_PATH)
else:
    # Fallback: clone from GitHub.
    # !git clone https://github.com/YOUR_USERNAME/arc-prize-2026.git /kaggle/working/repo
    # sys.path.insert(0, "/kaggle/working/repo")
    raise RuntimeError(
        f"{DATASET_PATH} not found. Attach the repo as a Kaggle Dataset or uncomment the git clone lines."
    )

In [ ]:
# 3. Load secrets into env vars (both SDKs read from env).
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["ARC_API_KEY"] = secrets.get_secret("ARC_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = secrets.get_secret("ANTHROPIC_API_KEY")

In [ ]:
# 4. Run the agent across all games the Arcade exposes.
import logging, json
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from agent.llm_agent import run_competition

scorecard = run_competition()
payload = scorecard.to_dict()
print(json.dumps(payload, indent=2, default=str))

In [ ]:
# 5. Persist the scorecard for Kaggle's grader.
out_path = "/kaggle/working/scorecard.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, default=str)
print("wrote", out_path)